# Zero-shot zaman serisi anomali modeli — eğitim

Bu defter `cagrigungor/tisan-havuz` (özel HF veri seti) üzerindeki gerçek arka planı ve `egitim_verisi_uretici.py`
sentetik verisini birleştirerek iki eksenli dikkat modelini eğitir, kalibre eder, benchmark'ta ölçer ve
HF'ye `trust_remote_code` ile yükler.

**Veri rolleri**

| Rol | Kaynak | Kullanım |
|---|---|---|
| Eğitim | LOTSA, HAI train + 20.07/21.03 test, CATS/ESA train, TEP, C-MAPSS, Bosch CNC, IMS, MIT-BIH, BATADAL train, LBNL, BGL, telekom, finans, otomotiv, sağlık arka planları | gerçek arka plan + gerçek anomali + sentetik anomali |
| Doğrulama | HAI 22.04/23.05 test, wind `labeled`, Pump, MetroPT-3, CATS val, ESA val, BATADAL test, ASD test | model seçimi, kalibrasyon (temperature) |
| Benchmark | NAB, SMAP/MSL, SMD, SKAB, UCR, PSM | **sadece** final rapor, eğitime girmez |

İlk eğitimden çıkan dersler (v1, 20k adım): sentetik hücre AUC-PR 0.52 / AUC-ROC 0.93, ama gerçek doğrulamada AUC-ROC ~0.6 ve
6000. adımdan sonra düşüş → sentetik stile aşırı uyum + `max` toplulaştırmanın 60 sütunlu HAI'de yanlış pozitif üretmesi.
Bu sürümde: sentetik payı 0.4 → 0.2, enjeksiyon 0.75 → 0.5, doğrulama 60k → 250k satır, satır toplulaştırma seçimi (top-k / noisy-OR),
checkpoint seçimi (AUC-PR + AUC-ROC)/2, `INIT_FROM` ile önceki ağırlıklardan devam.

Model boyutu `MODEL_SIZE` ile seçilir: `medium` (34M, **varsayılan**, H100'de ~1.5–2 saat), `small` (6.4M, T4'te ~80 dk), `base` (114M, A100/H100'de 6–8 saat).
Colab'da: Çalışma zamanı → GPU. HF token'ı **Secrets** panelinde `HF_TOKEN` olarak tanımla; deftere yapıştırma.

**v5 (TimeRCD bulguları):** bağlam 4096; denetimli aşamada bağlam-bağımlı sentetik korpus (`coupled`: DAG + ARX bağlaşımı, endojen enjeksiyon, etiket yayılımı) ana kaynak (P_SYNTHETIC 0.6), gerçek pencerelere enjeksiyon 0.3; VUS-PR (yaklaşık); Matrix Profile taban çizgisi.

**Mimari v4:** RoPE (Δt'ye göre gerçek zaman konumu), çok ölçekli girdi kanalları (ham, fark, 8 ve 64 satırlık yerel seviyeden sapma),
maskeli yeniden inşa ön eğitimi + yardımcı kayıp. Ablation: `pos_encoding="learned"`, `extra_channels=["diff"]`, `PRETRAIN_STEPS=0`, `AUX_RECON=0`.

**GPT-6 Astra düzeltmeleri:** Etiket güveni, satır/hücre kaybı, geçerli enjeksiyon,
paylaşılan curriculum sayacı, referanslı normalizasyon ve bağımsız kaynaklarda satır kalibrasyonu.
Yukarıdaki v1 ölçümleri eski akışa aittir; bu değişikliklerden sonra yeniden eğitim gerekir.


In [ ]:
import os, sys, subprocess
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # parçalanmaya karşı
if not os.path.exists("egitim_verisi_uretici.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/hasancagrigungor/tisan.git"], check=True)
    os.chdir("tisan")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.45", "huggingface_hub", "pyarrow", "pandas",
                "scikit-learn", "scipy", "tabulate", "matplotlib", "stumpy"], check=True)

from huggingface_hub import login, HfApi
try:
    from google.colab import userdata          # Colab Secrets
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("HF:", HfApi().whoami()["name"])

## Ayarlar

In [ ]:
SMOKE = os.environ.get("SMOKE") == "1"          # yerel duman testi: küçük model, birkaç adım

REPO_DATA  = "cagrigungor/tisan-havuz"
REPO_MODEL = "cagrigungor/anomali-small"
PUSH       = os.environ.get("PUSH", "0" if SMOKE else "1") == "1"

MODEL_SIZE = os.environ.get("MODEL_SIZE", "medium")   # small 6.4M (T4) | medium 34M (H100/A100, varsayılan) | base 114M
CFG = dict(patch=16, max_t=4096, max_ch=100, dropout=0.1,          # v5: bağlam 2048 → 4096 (uzun periyot, kalıcı anomali)
           extra_channels=["diff", "ms8", "ms64"],   # ablation: ["diff"] (çok ölçekli kapalı) | [] (yalın)
           pos_encoding="rope",                       # ablation: "learned"
           recon_head=True,                           # maskeli yeniden inşa başlığı (ön eğitim + yardımcı kayıp)
           use_types=os.environ.get("USE_TYPES", "1") == "1")   # tür başlığı: v5'te açık (yardımcı görev; çıktı yine 0/1, tür detect().events'te). Kapatmak: USE_TYPES=0
PRETRAIN_MASK = 0.3      # ön eğitimde maskelenen patch oranı
AUX_RECON     = 0.3      # denetimli eğitimde yardımcı yeniden inşa kaybı ağırlığı (0 = kapalı), %15 maske
if MODEL_SIZE == "small":
    CFG.update(d_model=256, n_layers=6, n_heads=8);   STEPS, BATCH, ACCUM, LR, WARMUP, EVAL_EVERY = 20_000, 16, 1, 3e-4, 500, 1_000; PRETRAIN_STEPS = 2_000
elif MODEL_SIZE == "medium":                          # H100 80 GB: batch 32 tek adımda sığar, checkpointing gerekmez
    CFG.update(d_model=512, n_layers=8, n_heads=8);   STEPS, BATCH, ACCUM, LR, WARMUP, EVAL_EVERY = 12_000, 32, 1, 3e-4, 800, 2_000; PRETRAIN_STEPS = 6_000   # 4096 bağlamda ~1.7 sn/adım → 12k ≈ 5.5 sa
else:
    CFG.update(d_model=768, n_layers=12, n_heads=12); STEPS, BATCH, ACCUM, LR, WARMUP, EVAL_EVERY = 30_000, 8, 2, 2e-4, 1_000, 2_000; PRETRAIN_STEPS = 8_000
TOKEN_BUDGET = {"small": 400_000, "medium": 150_000, "base": 60_000}[MODEL_SIZE]   # batch başına B×P×C üst sınırı; aşılırsa mikro-batch
GRAD_CKPT    = MODEL_SIZE == "base"    # bellek için aktivasyonları yeniden hesapla (~%30 yavaş); H100'de medium için gereksiz
P_SYNTHETIC  = 0.6        # v5: bağlam-bağımlı sentetik korpus ("coupled" alanı %40) denetimli aşamanın ana kaynağı (TimeRCD bulgusu)
UNKNOWN_WEIGHT = 0.25    # etiketsiz gerçek arka plan: kesin normal değil ama v4 teşhisi (çok kanallı gerçek veride Brier ~0.95, "her şey anomali") yanlış alarmı bastırmak için daha ağır negatif gerektiriyor; ablation: 0.05
P_INJECT     = 0.3        # gerçek pencerelere enjeksiyon azaltıldı: gerçek arka plan + enjeksiyon biçim ezberine yol açıyor
RUN_BASELINE = True       # Matrix Profile (MMPAD tarzı) taban çizgisi benchmark'ta hesaplanır
BASELINE_MAX_ROWS = 40_000
INIT_FROM    = os.environ.get("INIT_FROM", "")   # ör. "ckpt/best" veya HF repo: önceki eğitimden devam
CURRICULUM   = 0.6        # difficulty 0→1 bu oranda adımda tamamlanır
HOLDOUT_SECTORS = []      # leave-one-domain-out için ör. ["energy"]
MAX_CELLS_PER_SOURCE = 80_000_000   # RAM sınırı: kaynak başına satır×sütun
EVAL_MAX_ROWS = 150_000   # doğrulamada seri başına N satır (anomali yoksa ilk anomalinin etrafı)
SEED = 0

if SMOKE:
    CFG.update(d_model=64, n_layers=2, n_heads=4)
    STEPS, BATCH, ACCUM, EVAL_EVERY, WARMUP, PRETRAIN_STEPS = 30, 4, 1, 15, 5, 10
    MAX_CELLS_PER_SOURCE, EVAL_MAX_ROWS = 3_000_000, 6_000

import torch, numpy as np, random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
if AMP:
    torch.backends.cuda.matmul.allow_tf32 = True      # H100/A100: TF32 matmul
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
print("cihaz:", DEVICE, "| smoke:", SMOKE)

## Veri havuzu

In [ ]:
from huggingface_hub import snapshot_download
import pandas as pd
HAVUZ = "data/havuz"
if not os.path.exists(f"{HAVUZ}/katalog.csv"):
    snapshot_download(REPO_DATA, repo_type="dataset", local_dir=HAVUZ)
kat = pd.read_csv(f"{HAVUZ}/katalog.csv")
kat["benchmark"] = kat["benchmark"].fillna("")
kat["n_cells"] = kat.n_rows * kat.n_channels

is_bench = kat.benchmark != ""
# HAI: 20.07 ve 21.03 test dosyaları (gerçek saldırı etiketleri) eğitime; 22.04 ve 23.05 doğrulamaya
is_val = ((kat.source == "hai") & kat.series_id.str.contains("/test") & kat.series_id.str.contains("hai-22.04|hai-23.05"))          | (kat.series_id == "wind_gearbox/labeled") | (kat.source == "pump")          | kat.source.isin(["metropt"]) | kat.source.isin(["cats", "esa"]) & kat.series_id.str.endswith("/val")          | (kat.series_id == "batadal/test_dataset") | (kat.source == "asd") & kat.series_id.str.endswith("/test")
is_hold = kat.sector.isin(HOLDOUT_SECTORS)
kat["rol"] = np.select([is_bench, is_val, is_hold], ["benchmark", "val", "holdout"], "train")
# GPT-6 Astra: model seçimi ile kalibrasyon farklı KAYNAKLARDA yapılır.
_val_sources = sorted(kat.loc[kat.rol == "val", "source"].unique())
if len(_val_sources) < 2:
    raise ValueError("Model seçimi ve kalibrasyon için en az iki doğrulama kaynağı gerekli")
_split_rng = np.random.default_rng(SEED)
_cal_sources = _split_rng.choice(_val_sources, max(1, len(_val_sources) // 3), replace=False)
kat.loc[(kat.rol == "val") & kat.source.isin(_cal_sources), "rol"] = "calibration"
print(kat.groupby("rol").agg(seri=("series_id", "count"), satir_M=("n_rows", lambda x: round(x.sum() / 1e6, 1))))
print("eğitim sektörleri:", kat[kat.rol == "train"].sector.value_counts().to_dict())

In [ ]:
import pyarrow.parquet as pq

def load_series(rows, max_cells=None, rng=None):
    """Katalog satırlarını belleğe alır: {series_id: (t, X float32, labels int8)}. max_cells kaynak başına sınır."""
    rng = rng or np.random.default_rng(SEED)
    chosen = []
    for src, g in rows.groupby("source"):
        g = g.sample(frac=1, random_state=SEED)
        if max_cells:
            keep = g.n_cells.cumsum() <= max_cells
            keep.iloc[0] = True
            g = g[keep]
        chosen.append(g)
    chosen = pd.concat(chosen)
    out = {}
    for file, g in chosen.groupby("file"):
        ids = set(g.series_id)
        tbl = pq.read_table(f"{HAVUZ}/gercek/{file}", filters=[("series_id", "in", list(ids))]).to_pandas()
        tbl["series_id"] = tbl["series_id"].astype(str)
        meta = g.set_index("series_id")
        for sid, d in tbl.groupby("series_id", sort=False):
            T, k = int(meta.loc[sid, "n_rows"]), int(meta.loc[sid, "n_channels"])
            out[sid] = (d["timestamp"].to_numpy()[::k].astype(np.float64),
                        d["value"].to_numpy(np.float32).reshape(T, k),
                        d["label"].to_numpy(np.int8).reshape(T, k))
    return out

train_series = load_series(kat[kat.rol == "train"], MAX_CELLS_PER_SOURCE)
val_series   = load_series(kat[kat.rol == "val"])
cal_series   = load_series(kat[kat.rol == "calibration"])  # GPT-6 Astra
tot = sum(v[1].size for v in train_series.values())
print(f"eğitim: {len(train_series)} seri, {tot/1e6:.0f}M hücre | doğrulama: {len(val_series)} seri")

## Eğitim örnekleri

Her örnek ya tamamen sentetik (`make_sample`) ya da gerçek bir arka plan penceresi: rastgele uzunluk (20–2048, yarısı tam),
rastgele çözünürlük düşürme (blok ortalaması veya atlamalı: her 3./7. satır), rastgele düzensiz örnekleme, rastgele sütun alt kümesi, ardından üreticideki `generic_anomaly` ile enjekte edilen anomaliler.
Ön işleme `hf_model.modeling_anomali.prepare_window` — inference'taki `detect()` ile aynı kod.

In [ ]:
import math, warnings
from training_labels import supervision, label_confidence, normal_reference, row_targets
from egitim_verisi_uretici import safe_inject
warnings.filterwarnings("ignore", category=RuntimeWarning)
from egitim_verisi_uretici import make_sample, Ctx, generic_anomaly, TYPE_ID, MIN_T
import egitim_verisi_uretici as gen
from hf_model.modeling_anomali import prepare_window

MAX_T, MAX_CH, PATCH = CFG["max_t"], CFG["max_ch"], CFG["patch"]
gen.MAX_T = MAX_T                      # üretici ile aynı pencere

kat_train = kat[kat.rol == "train"].set_index("series_id").loc[list(train_series)]
_sectors = sorted(kat_train.sector.unique())
_by_sector = {s: kat_train[kat_train.sector == s] for s in _sectors}
_sector_w = {s: np.sqrt(g.n_rows.to_numpy()) for s, g in _by_sector.items()}

def _pick_series(rng):
    s = _sectors[rng.integers(len(_sectors))]              # sektörler dengeli
    g, w = _by_sector[s], _sector_w[s]
    return g.index[rng.choice(len(g), p=w / w.sum())]

def _downsample(t, X, lab, f, mode="mean"):
    # mean: blok ortalaması (10 sn -> 1 dk). stride: her f. satır (her 3 günde bir, sadece cumalar gibi;
    # takvim özelliği olmadığı için model bunu sabit delta-t olarak görür)
    n = len(t) // f
    if mode == "stride":
        return t[:n * f:f], X[:n * f:f], lab[:n * f:f]
    Xb = X[:n * f].reshape(n, f, -1)
    return t[:n * f:f], np.nanmean(Xb, axis=1), lab[:n * f].reshape(n, f, -1).max(1)

def _random_drop(rng, t, X, lab, keep_frac):
    # düzensiz örnekleme: satırların bir kısmı rastgele atılır, delta-t değişken olur (etiketler korunur)
    keep = np.sort(rng.choice(len(t), max(MIN_T, int(len(t) * keep_frac)), replace=False))
    return t[keep], X[keep], lab[keep]

def real_item(rng, difficulty):
    for _ in range(20):
        sid = _pick_series(rng)
        t, X, lab = train_series[sid]
        meta = kat_train.loc[sid]
        confidence = label_confidence(meta)
        level = meta.label_level
        f = int(rng.choice([1, 1, 1, 2, 3, 4, 7, 8, 12]))
        if len(t) // f < MIN_T * 2:
            f = 1
        if f > 1:
            t, X, lab = _downsample(t, X, lab, f, mode="stride" if rng.random() < 0.5 else "mean")
        if rng.random() < 0.15 and len(t) > MIN_T * 3:                  # düzensiz örnekleme
            t, X, lab = _random_drop(rng, t, X, lab, rng.uniform(0.5, 0.9))
        n, kc = X.shape
        L = MAX_T if rng.random() < 0.5 else int(rng.integers(MIN_T, MAX_T + 1))
        L = min(L, n)
        a = int(rng.integers(0, n - L + 1))
        k = min(kc, max(1, int(round(math.exp(rng.uniform(0, math.log(min(MAX_CH, kc)) if kc > 1 else 0))))))
        # GPT-6 Astra: satır pozitifinde arızalı kanal bilinmez; mümkünse tüm kanalları göster.
        if level == "row":
            k = min(kc, MAX_CH)
        cols = rng.choice(kc, k, replace=False)
        original_rows = row_targets(lab[a:a + L])
        ref = normal_reference(X, lab, a) if confidence == 1.0 else None
        Xw, lw, tw = X[a:a + L][:, cols].astype(np.float64), lab[a:a + L][:, cols].copy(), t[a:a + L].copy()
        ok = ~np.isnan(Xw).all(0) & (np.nanstd(Xw, axis=0) > 0)
        if not ok.any():
            continue
        Xw, lw = Xw[:, ok], lw[:, ok]
        # GPT-6 Astra: tümü anomalili pencere ancak geçmiş doğrulanmış normal referansla öğrenilir.
        dense = (lw == 1).any(1).mean() > 0.6
        if dense and ref is None:
            continue
        ref = ref[:, cols][:, ok] if ref is not None and (dense or rng.random() < 0.25) else None
        Xw = gen_fill(Xw)
        k = Xw.shape[1]
        # GPT-6 Astra: bilinmeyen, zayıf, satır ve hücre denetimini ayrı taşı.
        real_lab, weights, row_lab, row_w = supervision(
            lw, level, confidence, UNKNOWN_WEIGHT, rows=original_rows,
            row_positive_visible=(k == kc))
        types = np.full_like(real_lab, -1)
        blocked = np.broadcast_to((original_rows == 1)[:, None], real_lab.shape).copy()
        if rng.random() < globals().get("_INJECT", P_INJECT):     # ön eğitim akışı _INJECT=0 yapar
            keep = None
            if rng.random() < gen.MISSING_DATA_RATIO and L > 40:
                gap = int(rng.integers(5, min(100, L // 4)))
                gi = int(rng.integers(L // 10, L - gap - 1))
                keep = np.r_[0:gi, gi + gap:L]
                Xw, tw, real_lab, types = Xw[keep], tw[keep], real_lab[keep], types[keep]
                weights, row_lab, row_w, blocked = weights[keep], row_lab[keep], row_w[keep], blocked[keep]
            ctx = Ctx(rng, Xw, tw, ["unknown"] * k, np.zeros(k, dtype=int), difficulty, {}, None)
            if keep is not None:
                ctx.mark(gi, gi + 1, list(range(k)), "missing_data")
            for _ in range(int(rng.integers(1, 4))):
                for _attempt in range(5):
                    if safe_inject(ctx, generic_anomaly, blocked=blocked):
                        break
            Xw = ctx.X
            inj = ctx.labels == 1
            real_lab = np.where(inj, 1, real_lab).astype(np.int8)
            weights[inj] = 1.0
            # Enjekte edilen hücre zaten denetleniyor; aynı satırı ikinci kez sayma.
            row_lab[inj.any(1)] = 1
            row_w[inj.any(1)] = 0
            types = np.where(inj, ctx.types, types)
        return pack(tw, Xw, real_lab, types, weights, row_lab, row_w, reference=ref)
    return synthetic_item(rng, difficulty)

def gen_fill(X):
    from hf_model.modeling_anomali import fill_nan
    return fill_nan(X)

def pack(t, X, labels, types, weights=None, row_labels=None, row_weights=None, reference=None):
    T, k = X.shape
    values, dtf, tm, cm = prepare_window(t, X, MAX_T, MAX_CH, reference=reference)
    lab = np.zeros((MAX_T, MAX_CH), dtype=np.int8); lab[:T, :k] = labels
    typ = np.zeros((MAX_T, MAX_CH), dtype=np.int8); typ[:T, :k] = types
    # GPT-6 Astra: dolgu hücrelerinin/satırlarının güven ağırlığı sıfırdır.
    w = np.zeros((MAX_T, MAX_CH), dtype=np.float32)
    w[:T, :k] = (labels >= 0) if weights is None else weights
    rl = np.full(MAX_T, -1, dtype=np.int8)
    rw = np.zeros(MAX_T, dtype=np.float32)
    if row_labels is not None:
        rl[:T] = row_labels
        rw[:T] = row_weights
    return dict(values=values, delta_t=dtf, time_mask=tm, channel_mask=cm, labels=lab,
                types=typ, label_weights=w, row_labels=rl, row_weights=rw)

def synthetic_item(rng, difficulty):
    for _ in range(10):
        s = make_sample(rng, difficulty=difficulty)
        if np.isfinite(s["raw"]).all():
            break
    # üretici kendi normalize eder; NaN yok. Aynı ön işleme için ham matristen yeniden paketle.
    T, k = s["meta"]["T"], s["meta"]["k"]
    t, X = s["raw"][:, 0], s["raw"][:, 1:]
    labels, types = s["labels"][:T, :k], s["types"][:T, :k]
    # GPT-6 Astra: normal bağlam pencere dışında kaldığında da referanslı eğitim örneği üret.
    ref = None
    pos = np.flatnonzero(labels.any(1))
    if len(pos) and pos[0] >= MIN_T and T - pos[0] >= MIN_T and rng.random() < 0.25:
        start = int(rng.integers(pos[0], T - MIN_T + 1))
        ref = X[:pos[0]].copy()
        t, X, labels, types = t[start:], X[start:], labels[start:], types[start:]
    return pack(t, X, labels, types, reference=ref)

class TrainStream(torch.utils.data.IterableDataset):
    def __init__(self, shared_step, pretrain=False):
        self.shared_step = shared_step  # GPT-6 Astra: worker kopyaları aynı adım sayacını okur.
        self.pretrain = pretrain        # ön eğitim: yalnızca gerçek pencereler, enjeksiyon yok, etiket kullanılmaz
    def __iter__(self):
        wi = torch.utils.data.get_worker_info()
        rng = np.random.default_rng([SEED, wi.id if wi else 0, int(torch.initial_seed()) % 2**31])
        if self.pretrain:
            globals()["_INJECT"] = 0.0
            while True:
                yield real_item(rng, 0.0)
        while True:
            d = min(1.0, self.shared_step.value / max(1, STEPS * CURRICULUM))
            yield synthetic_item(rng, d) if rng.random() < P_SYNTHETIC else real_item(rng, d)

def collate(items):
    b = {k: torch.from_numpy(np.stack([it[k] for it in items])) for k in items[0]}
    # dolgu sağda: geçerli en uzun T (patch katı) ve en çok sütuna kırp → hesap tasarrufu
    T_eff = int(math.ceil(int(b["time_mask"].sum(1).max()) / PATCH) * PATCH)
    C_eff = int(b["channel_mask"].sum(1).max())
    for k in ("values", "labels", "types", "label_weights"):
        b[k] = b[k][:, :T_eff, :C_eff].contiguous()
    b["delta_t"], b["time_mask"] = b["delta_t"][:, :T_eff], b["time_mask"][:, :T_eff]
    b["channel_mask"] = b["channel_mask"][:, :C_eff]
    for key in ("row_labels", "row_weights"):
        b[key] = b[key][:, :T_eff]
    return b

_rng = np.random.default_rng(1)
for _ in range(3):
    it = real_item(_rng, 0.5)
    print("gerçek:", int(it["time_mask"].sum()), "satır", int(it["channel_mask"].sum()), "sütun, anomali hücre", int(it["labels"].sum()))
it = synthetic_item(_rng, 0.5); print("sentetik:", int(it["time_mask"].sum()), "satır", int(it["channel_mask"].sum()), "sütun")

## Model

In [ ]:
from hf_model.configuration_anomali import AnomaliConfig
from hf_model.modeling_anomali import AnomaliModel

config = AnomaliConfig(**CFG)
if INIT_FROM:
    model = AnomaliModel.from_pretrained(INIT_FROM).to(DEVICE)
    print("ağırlıklar yüklendi:", INIT_FROM)
else:
    model = AnomaliModel(config).to(DEVICE)
model.gradient_checkpointing = GRAD_CKPT
print(f"parametre: {sum(p.numel() for p in model.parameters())/1e6:.1f}M | etkin batch: {BATCH * ACCUM} | checkpointing: {GRAD_CKPT}")

## Değerlendirme

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

from hf_model.modeling_anomali import aggregate_rows, calibrate_rows

def vus_pr(y, s, buffers=(0, 16, 64, 256)):
    """VUS-PR (yaklaşık): etiketler farklı tolerans genişlikleriyle iki yana genişletilir, AP ortalanır.
    TSB-AD'nin VUS-PR'ı gibi sınır hassasiyetini azaltır; point-adjust değildir."""
    y = np.asarray(y).astype(bool); out = []
    for b in buffers:
        if b == 0:
            yb = y
        else:
            idx = np.flatnonzero(y); yb = y.copy()
            for i0 in idx[np.r_[True, np.diff(idx) > 1]]:
                yb[max(0, i0 - b):i0] = True
            for i1 in idx[np.r_[np.diff(idx) > 1, True]]:
                yb[i1:i1 + b + 1] = True
        out.append(average_precision_score(yb, s))
    return float(np.mean(out))

_cell_cache = {}          # sid -> (prob, y): toplulaştırma karşılaştırması için

def eval_real(series, max_rows=None, batch_size=4, agg="topk", topk=3, cache=False):
    """Etiketli gerçek serilerde satır bazında AUC-PR / AUC-ROC / en iyi F1 (point-adjust YOK)."""
    model.eval()
    rows = []
    for sid, (t, X, lab) in series.items():
        if max_rows and len(t) > max_rows:
            first = np.argmax((lab == 1).any(1)) if (lab == 1).any() else 0
            a = 0 if first < max_rows * 0.8 else max(0, int(first - max_rows // 2))     # ör. MetroPT: ilk arıza 660k. satırda
            t, X, lab = t[a:a + max_rows], X[a:a + max_rows], lab[a:a + max_rows]
        target = row_targets(lab)
        known = target >= 0
        y = target[known].astype(int)
        if y.sum() == 0 or y.sum() == len(y):
            continue
        if cache and sid in _cell_cache:
            prob = _cell_cache[sid][0]
        else:
            prob, _ = model.score_matrix(t, X.astype(np.float64), batch_size=batch_size)
            if cache:
                _cell_cache[sid] = (prob, target)
        s = np.nan_to_num(aggregate_rows(prob, agg, topk), nan=0.0)[known]
        s = calibrate_rows(s, model.config.row_temperature, model.config.row_bias)
        p, r, _ = precision_recall_curve(y, s)
        rows.append(dict(series_id=sid, source=sid.split("/")[0], n=len(y), anom=float(y.mean()),
                         auc_pr=average_precision_score(y, s), vus_pr=vus_pr(y, s), auc_roc=roc_auc_score(y, s),
                         brier=float(np.mean((s - y) ** 2)),
                         nll=float(-np.mean(y * np.log(np.clip(s, 1e-7, 1)) + (1-y) * np.log(np.clip(1-s, 1e-7, 1)))),
                         best_f1=float(np.max(2 * p * r / np.maximum(p + r, 1e-9)))))
    model.train()
    if AMP: torch.cuda.empty_cache()
    return pd.DataFrame(rows)

_vr = np.random.default_rng(123)
SYN_VAL = [synthetic_item(_vr, 0.7) for _ in range(8 if SMOKE else 200)]

@torch.no_grad()
def eval_synthetic(items=SYN_VAL, batch_size=4):
    model.eval()
    ys, ps, ty, tp = [], [], [], []
    for i in range(0, len(items), batch_size):
        b = {k: v.to(DEVICE) for k, v in collate(items[i:i + batch_size]).items()}
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=AMP):
            out = model(b["values"], b["delta_t"], b["time_mask"], b["channel_mask"])
        out = {k: v.float() for k, v in out.items()}
        valid = (b["time_mask"][:, :, None] & b["channel_mask"][:, None, :])
        ys.append(b["labels"][valid].cpu().numpy()); ps.append(torch.sigmoid(out["logits"])[valid].float().cpu().numpy())
        tm = valid & (b["labels"] > 0) & (b["types"] > 0)
        ty.append(b["types"][tm].cpu().numpy()); tp.append(out["type_logits"][tm].argmax(-1).cpu().numpy())
        del out, b
    model.train()
    if AMP: torch.cuda.empty_cache()
    y, p = np.concatenate(ys), np.concatenate(ps)
    res = dict(cell_auc_pr=average_precision_score(y, p), cell_auc_roc=roc_auc_score(y, p))
    if model.config.use_types and len(ty):
        res["type_acc"] = float((np.concatenate(ty) == np.concatenate(tp)).mean())
    return res

print(eval_synthetic())

## Eğitim

## Ön eğitim: maskeli yeniden inşa (MOMENT/PatchTST tarzı)

Etiketsiz gerçek pencerelerde patch'lerin %30'u mask token ile gizlenir, model normalize değerleri yeniden inşa eder.
"Bu seride normal ne?" sorusu etiket olmadan öğrenilir; anomali eğitimi bu ağırlıklardan başlar.
`PRETRAIN_STEPS=0` ile atlanır (ablation).

In [ ]:
import time, math
from torch.optim.lr_scheduler import LambdaLR
import multiprocessing as mp

def random_patch_mask(b, frac):
    B, T, C = b["values"].shape
    P = T // PATCH
    valid = b["time_mask"].reshape(B, P, PATCH).any(-1)[:, :, None] & b["channel_mask"][:, None, :]
    return (torch.rand(B, P, C, device=b["values"].device) < frac) & valid

def micro_batches(b, budget=None):
    """Bellek koruması: B×P×C token sayısı bütçeyi aşarsa batch'i parçalara böler (gradyan biriktirilir)."""
    budget = budget or TOKEN_BUDGET
    B, T, C = b["values"].shape
    n = max(1, math.ceil(B * (T // PATCH) * C / budget))
    step = math.ceil(B / n)
    for j in range(0, B, step):
        yield {k: v[j:j + step] for k, v in b.items()}, n

if PRETRAIN_STEPS > 0 and not INIT_FROM:
    pre_loader = torch.utils.data.DataLoader(TrainStream(mp.Value("q", 0), pretrain=True), batch_size=BATCH, collate_fn=collate,
                                             num_workers=0 if SMOKE else 4, persistent_workers=not SMOKE, prefetch_factor=None if SMOKE else 4)
    pre_opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05, betas=(0.9, 0.98))
    pre_sched = LambdaLR(pre_opt, lambda s: min(1.0, (s + 1) / max(1, WARMUP // 2)) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / PRETRAIN_STEPS))))
    model.train(); t0 = time.time(); run = 0.0; step = 0; micro = 0
    for batch in pre_loader:
        b = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        tot = 0.0
        for sb, n in micro_batches(b):
            mask = random_patch_mask(sb, PRETRAIN_MASK)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=AMP):
                out = model(sb["values"], sb["delta_t"], sb["time_mask"], sb["channel_mask"], mask_patches=mask)
            if "loss" not in out:
                continue
            (out["loss"] / (ACCUM * n)).backward(); tot += out["loss"].item() / n
            del out
        micro += 1
        if micro % ACCUM != 0:
            continue
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        pre_opt.step(); pre_sched.step(); pre_opt.zero_grad(set_to_none=True); step += 1
        run = 0.98 * run + 0.02 * tot if step > 1 else tot
        if step % 100 == 0:
            print(f"ön eğitim {step:6d} | yeniden inşa kaybı {run:.4f} | lr {pre_sched.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s")
        if step >= PRETRAIN_STEPS:
            break
    os.makedirs("ckpt", exist_ok=True); model.save_pretrained("ckpt/pretrain")
    print(f"ön eğitim bitti: {step} adım, son kayıp {run:.4f}; ckpt/pretrain")
    del pre_loader, pre_opt
    if AMP: torch.cuda.empty_cache()
else:
    print("ön eğitim atlandı" + (f" (INIT_FROM={INIT_FROM})" if INIT_FROM else ""))

In [ ]:
import time
from torch.optim.lr_scheduler import LambdaLR

import multiprocessing as mp
state = mp.Value("q", 0)  # GPT-6 Astra: süreçler arası paylaşılan curriculum adımı
def difficulty():
    return min(1.0, state.value / max(1, STEPS * CURRICULUM))

loader = torch.utils.data.DataLoader(TrainStream(state), batch_size=BATCH, collate_fn=collate,
                                     num_workers=0 if SMOKE else 4, persistent_workers=not SMOKE, prefetch_factor=None if SMOKE else 4)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05, betas=(0.9, 0.98))
sched = LambdaLR(opt, lambda s: min(1.0, (s + 1) / WARMUP) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / STEPS))))
micro = 0

wandb = None
if os.environ.get("WANDB_API_KEY") and not SMOKE:
    import wandb as _wb; wandb = _wb; wandb.init(project="tisan-anomali", config={**CFG, "steps": STEPS, "batch": BATCH, "lr": LR})

best, t0, run_loss = -1.0, time.time(), 0.0
os.makedirs("ckpt", exist_ok=True)
model.train()
for batch in loader:
    b = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
    loss_val = 0.0
    for sb, n in micro_batches(b):                                     # token bütçesine göre otomatik parçalama
        aux = random_patch_mask(sb, 0.15) if AUX_RECON > 0 else None   # yardımcı yeniden inşa (MOMENT tarzı düzenleyici)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=AMP):
            out = model(**sb, mask_patches=aux, recon_weight=AUX_RECON)
        (out["loss"] / (ACCUM * n)).backward(); loss_val += out["loss"].item() / n
        del out
    micro += 1
    if micro % ACCUM != 0:
        continue
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
    state.value += 1; s = state.value
    run_loss = 0.98 * run_loss + 0.02 * loss_val if s > 1 else loss_val
    if s % 50 == 0:
        print(f"adım {s:6d} | kayıp {run_loss:.4f} | zorluk {difficulty():.2f} | lr {sched.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s")
        if wandb: wandb.log({"loss": run_loss, "difficulty": difficulty(), "lr": sched.get_last_lr()[0]}, step=s)
    if s % EVAL_EVERY == 0 or s == STEPS:
        syn = eval_synthetic()
        real = eval_real(val_series, max_rows=EVAL_MAX_ROWS)
        score = float((real.auc_pr.mean() + real.auc_roc.mean()) / 2) if len(real) else syn["cell_auc_pr"]
        print(f"  ↳ sentetik {syn} | gerçek AUC-PR {real.auc_pr.mean():.3f} AUC-ROC {real.auc_roc.mean():.3f} (n={len(real)})")
        if wandb: wandb.log({**{f"syn/{k}": v for k, v in syn.items()}, "val/auc_pr": real.auc_pr.mean(), "val/auc_roc": real.auc_roc.mean()}, step=s)
        if score > best:
            best = score; model.save_pretrained("ckpt/best"); print("  ↳ en iyi model kaydedildi")
    if s >= STEPS:
        break
print("bitti; en iyi doğrulama skoru (AUC-PR+AUC-ROC)/2:", round(best, 4))

## Teşhis: seri bazında doğrulama ve satır toplulaştırma

Satır skoru = hücre skorlarının birleşimi. `max` çok sütunlu seride (HAI: 60 sütun) tek bir yanlış pozitif hücreyi satıra taşır.
Aynı checkpoint'le üç yöntem karşılaştırılır, en iyisi `config.row_agg` olarak modele yazılır; eğitim gerekmez.

In [ ]:
model = AnomaliModel.from_pretrained("ckpt/best").to(DEVICE).eval()
_cell_cache.clear()
karsilastirma = {}
for agg, k in [("max", 1), ("topk", 3), ("topk", 5), ("noisy_or", 1)]:
    r = eval_real(val_series, max_rows=EVAL_MAX_ROWS, agg=agg, topk=k, cache=True)
    karsilastirma[f"{agg}{k if agg == 'topk' else ''}"] = r.assign(yontem=f"{agg}{k if agg == 'topk' else ''}")
ozet = pd.concat(karsilastirma.values()).groupby("yontem").agg(auc_pr=("auc_pr", "mean"), auc_roc=("auc_roc", "mean"), best_f1=("best_f1", "mean")).round(3)
print(ozet)
en_iyi = ozet.assign(s=(ozet.auc_pr + ozet.auc_roc) / 2).s.idxmax()
model.config.row_agg = "noisy_or" if en_iyi == "noisy_or" else ("max" if en_iyi == "max" else "topk")
model.config.row_topk = int(en_iyi[4:]) if en_iyi.startswith("topk") else 3
print("seçilen:", model.config.row_agg, model.config.row_topk)
print()
print("seri bazında (seçilen yöntem):")
print(karsilastirma[en_iyi].drop(columns="yontem").sort_values("auc_roc").round(3).to_string(index=False))
model.save_pretrained("ckpt/best")

In [ ]:
import matplotlib.pyplot as plt
# en kötü ve en iyi doğrulama serisi: skor vs etiket
tab = karsilastirma[en_iyi].sort_values("auc_roc")
for sid in [tab.series_id.iloc[0], tab.series_id.iloc[-1]]:
    prob, y = _cell_cache[sid]
    s = aggregate_rows(prob, model.config.row_agg, model.config.row_topk)
    n = min(len(s), 100_000)
    fig, ax = plt.subplots(2, 1, figsize=(14, 4), sharex=True)
    ax[0].plot(s[:n], lw=0.5); ax[0].set_ylabel("satır skoru"); ax[0].set_title(sid)
    ax[1].fill_between(np.arange(n), 0, y[:n], color="red", alpha=0.5); ax[1].set_ylabel("etiket")
    plt.tight_layout(); plt.show()
    # hangi sütunlar en çok alarm veriyor?
    fp = (prob[:n][y[:n] == 0] > 0.7).mean(0)
    print(sid, "normal satırlarda en çok alarm veren sütunlar (yanlış pozitif oranı):", {int(i): round(float(fp[i]), 3) for i in np.argsort(fp)[-5:][::-1]})

## Kalibrasyon — GPT-6 Astra
Model seçiminde kullanılmayan kaynakların gerçek satır etiketleriyle, toplulaştırma sonrasında temperature + bias öğrenilir.
Sentetik veriler ve bilinmeyen etiketler kalibrasyona girmez. Benchmark üzerinde Brier/NLL raporlanır;
bu işlem farklı sektörlerde “0.9 = %90” garantisi vermez. Normal referanslı kullanım ayrıca doğrulanmalıdır.


In [ ]:
# GPT-6 Astra: hücre etiketini bilmeden hücre kalibrasyonu yapılmaz.
model.eval()
model.config.temperature = 1.0
model.config.row_temperature, model.config.row_bias = 1.0, 0.0
cal_scores, cal_targets = [], []
for sid, (t, X, lab) in cal_series.items():
    target = row_targets(lab)
    known = target >= 0
    if not known.any():
        continue
    prob, _ = model.score_matrix(t, X.astype(np.float64))
    score = aggregate_rows(prob, model.config.row_agg, model.config.row_topk)
    cal_scores.append(score[known]); cal_targets.append(target[known])
if not cal_scores:
    raise ValueError("Kalibrasyon için bilinen gerçek etiket yok")
p_cal = np.concatenate(cal_scores)
y_cal = np.concatenate(cal_targets)
if len(np.unique(y_cal)) < 2:
    raise ValueError("Kalibrasyon normal ve anomali sınıflarının ikisini de gerektirir")
# Doğal sınıf oranı korunur; sınıf dengesi için yeniden örnekleme yapılmaz.
import torch.nn.functional as F
z = torch.tensor(np.log(np.clip(p_cal, 1e-7, 1-1e-7)) - np.log1p(-np.clip(p_cal, 1e-7, 1-1e-7)), dtype=torch.float64)
y = torch.tensor(y_cal, dtype=torch.float64)
log_T = torch.zeros((), dtype=torch.float64, requires_grad=True)
bias = torch.zeros((), dtype=torch.float64, requires_grad=True)
cal_opt = torch.optim.LBFGS([log_T, bias], lr=0.5, max_iter=100, line_search_fn="strong_wolfe")
def calibration_closure():
    cal_opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(z / log_T.clamp(-3, 3).exp() + bias.clamp(-10, 10), y)
    loss.backward()
    return loss
cal_opt.step(calibration_closure)
model.config.row_temperature = float(log_T.detach().clamp(-3, 3).exp())
model.config.row_bias = float(bias.detach().clamp(-10, 10))
print("satır kalibrasyonu:", model.config.row_temperature, model.config.row_bias)
print("kalibrasyon uyum NLL (bağımsız test değildir):", float(calibration_closure().detach()))
model.save_pretrained("ckpt/best")


## Benchmark (sadece rapor)
NAB, SMAP/MSL, SMD, SKAB. Eğitimde hiç görülmedi. Point-adjust uygulanmaz.

In [ ]:
bench_series = load_series(kat[kat.rol == "benchmark"])
bench = eval_real(bench_series, max_rows=EVAL_MAX_ROWS if SMOKE else None, agg=model.config.row_agg, topk=model.config.row_topk)
tablo = bench.groupby("source").agg(seri=("series_id", "count"), auc_pr=("auc_pr", "mean"), vus_pr=("vus_pr", "mean"), auc_roc=("auc_roc", "mean"), best_f1=("best_f1", "mean"), brier=("brier", "mean"), nll=("nll", "mean")).round(3)
print(tablo)
bench.to_csv("ckpt/benchmark.csv", index=False)

# --- Taban çizgisi: Matrix Profile (MMPAD'ın çekirdeği; TSB-AD lider tablosunda 1.) ---
if RUN_BASELINE:
    import stumpy
    def mp_scores(X, m=64):
        X = np.nan_to_num(X.astype(np.float64)); n, k = X.shape
        m = int(min(m, max(8, n // 20)))
        if k == 1:
            mp = stumpy.stump(X[:, 0], m)[:, 0].astype(float)
        else:
            cols = np.argsort(-X.std(0))[:8]                     # en oynak 8 sütun (mstump maliyeti)
            mp = np.nan_to_num(stumpy.mstump(X[:, cols].T, m)[0]).mean(0)
        s = np.zeros(n); s[m // 2:m // 2 + len(mp)] = mp        # pencere ortasına hizala
        s[:m // 2] = s[m // 2]; s[m // 2 + len(mp):] = s[m // 2 + len(mp) - 1]
        return pd.Series(s).rolling(m, min_periods=1, center=True).mean().to_numpy()   # hareketli ortalama (MMPAD)
    brows = []
    for sid, (t, X, lab) in list(bench_series.items())[: (2 if SMOKE else None)]:
        target = row_targets(lab); n = min(len(t), BASELINE_MAX_ROWS)
        known = target[:n] >= 0; y = target[:n][known].astype(int)
        if len(y) == 0 or y.sum() == 0 or y.sum() == len(y):
            continue
        s = mp_scores(X[:n])[known]
        brows.append(dict(series_id=sid, source=sid.split("/")[0], auc_pr=average_precision_score(y, s), vus_pr=vus_pr(y, s), auc_roc=roc_auc_score(y, s)))
    baseline = pd.DataFrame(brows)
    if len(baseline):
        tablo_mp = baseline.groupby("source").agg(seri=("series_id", "count"), mp_auc_pr=("auc_pr", "mean"), mp_vus_pr=("vus_pr", "mean"), mp_auc_roc=("auc_roc", "mean")).round(3)
        print("\nMatrix Profile taban çizgisi:"); print(tablo_mp)
        print("\nmodel − MP (VUS-PR):"); print((tablo["vus_pr"] - tablo_mp["mp_vus_pr"]).dropna().round(3).to_string())
        baseline.to_csv("ckpt/benchmark_mp.csv", index=False)

## Hugging Face'e yükleme (remote code)

In [ ]:
from hf_model.configuration_anomali import AnomaliConfig
from hf_model.modeling_anomali import AnomaliModel
AnomaliConfig.register_for_auto_class()
AnomaliModel.register_for_auto_class("AutoModel")

card = f"""---
license: apache-2.0
tags: [time-series, anomaly-detection, zero-shot]
---
# anomali-small

Zero-shot zaman serisi anomali modeli. Girdi: `(T, 1+k)` matris, ilk sütun zaman damgası, 1–100 değer sütunu.
Eğitim/etiket/ayar gerektirmez. {sum(p.numel() for p in model.parameters())/1e6:.1f}M parametre, iki eksenli dikkat (zaman → sütun).

```python
from transformers import AutoModel
model = AutoModel.from_pretrained("{REPO_MODEL}", trust_remote_code=True)
y = model.predict(matris)             # (T,) 0/1: satırda anomali var mı — tek gereken bu
sonuc = model.detect(matris)          # ayrıntı: .row_scores, .cell_scores (hangi sütun), .events, .to_dataframe(), .plot()
```

## Benchmark (eğitimde görülmedi, point-adjust yok)
{tablo.to_markdown()}

## Sınırlar
- Geleceği tahmin etmez; pencere içindeki sapmaları işaretler; çevrimiçi gecikmesiz tespit garantisi yoktur. < 20 satırda robust z-skoruna düşer.
- Çıktı alan bilgisi gerektirmez: finans, sensör, siber güvenlik ya da bilinmeyen veri aynı yoldan geçer. Satır skorları ayrı gerçek kaynaklarda toplulaştırma sonrası kalibre edilmiştir (T={model.config.row_temperature:.2f}, bias={model.config.row_bias:.2f}); `{model.config.row_agg}` toplulaştırması. Yeni sektörlerde olasılık garantisi yoktur.
- GPT-6 Astra: satır/hücre etiketleri ayrılır, etiketsiz arka plan düşük ağırlıklıdır; zayıf bozulma etiketleri kesin etiket sayılmaz.
- Kalıcı seviye değişimleri için `normal_reference` ile doğrulanmış normal değer matrisi verilebilir; otomatik normal referans seçimi ve olay sürekliliği henüz yoktur.
- Eğitim verisi: LOTSA, HAI, açık imalat/enerji SCADA setleri + çok alanlı sentetik anomaliler.
"""
if PUSH:
    model.push_to_hub(REPO_MODEL, private=True, commit_message="eğitim çıktısı")
    HfApi().upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=REPO_MODEL, repo_type="model")
    print("yüklendi:", f"https://huggingface.co/{REPO_MODEL}")
else:
    model.save_pretrained("ckpt/hub"); open("ckpt/hub/README.md", "w").write(card); print("PUSH=0: ckpt/hub altına kaydedildi")

## Kullanım örneği

In [ ]:
from transformers import AutoModel
m = AutoModel.from_pretrained(REPO_MODEL if PUSH else "ckpt/hub", trust_remote_code=True).to(DEVICE).eval()
sid, (t, X, lab) = next(iter(val_series.items()))
n = min(len(t), 6000)
y01 = m.predict(np.column_stack([t[:n], X[:n, :8]]))          # birincil API: (T,) 0/1
print("predict:", y01.shape, "anomali satır:", int(y01.sum()))
sonuc = m.detect(np.column_stack([t[:n], X[:n, :8]]))
print(sid, sonuc, "| gerçek anomali satırı:", int((lab[:n] == 1).any(1).sum()))
for e in sonuc.events[:5]:
    print(e)
try:
    sonuc.plot()
except Exception as ex:
    print("çizim atlandı:", ex)